Week 4 Updates:
- Modified output prints to be more elaborate. Added verbose flag to turn on/off prints as needed. The modified prints are shown in the playground section of this notebook.
- Created a thorough comparison of the performance of all the models againts the annotated data set. Barplots were averaged over all metrics for each model https://colab.research.google.com/drive/1jAWRG8av4D_eMNwDuMOrPvC1I5kO6hbA?usp=sharing
- Replaced hardcoded variables in the previously created modules with input variables: output directory, maximum number of users to index, name of column with user ids in the csv file, name of column with objectives in the csv file
- Created prelimenary flask server with the following endpoints:
    - /upload : To upload the csv file with the user objectves.
    - /models : To get the list of supported models by the system.
    - /metrics: To get the list of supported metrics by the system.
    - /generate_models : generates embedding models depending on the passed arguments
    - /get_sentence: Returns the user objective of the given user id
    - /get_nearest: Returns the most similar users depending on the passed arguments.
    The server file is shown in the bottom most section of this notebook. The code itself is in Server/server.py in the gitlab repo.

Jump to the playground section directly to view the changes in the output prints

## Install Required Libraries

In [0]:
%%capture
# Install the latest Tensorflow version.
!pip3 install --upgrade tensorflow-gpu
# Install TF-Hub.
!pip3 install tensorflow==2.0
!pip3 install tensorflow-hub
!pip3 install bert-for-tf2
!pip3 install sentencepiece

## Import Required Libraries to Python

In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import tensorflow_hub as hub
import tensorflow as tf 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sentencepiece as spm
import tokenization
import time
import bert
import h5py
FullTokenizer = bert.bert_tokenization.FullTokenizer

from tensorflow.keras.models import Model
from sklearn.preprocessing import minmax_scale
from tabulate import tabulate

from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial import distance
from numpy import linalg as LA

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of tensorflow.summary failed: Traceback (most recent call last):
  File "/usr/local/lib/python3.6/dist-packages/IPython/extensions/autoreload.py", line 247, in check
    superreload(m, reload, self.old_objects)
RecursionError: maximum recursion depth exceeded while calling a Python object
]
[autoreload of tensorflow.estimator failed: Traceback (most recent call last):
  File "/usr/local/lib/python3.6/dist-packages/IPython/extensions/autoreload.py", line 247, in check
    superreload(m, reload, self.old_objects)
RecursionError: maximum recursion depth exceeded while calling a Python object
]
[autoreload of tensorflow_estimator failed: Traceback (most recent call last):
  File "/usr/local/lib/python3.6/dist-packages/IPython/extensions/autoreload.py", line 247, in check
    superreload(m, reload, self.old_objects)
RecursionError: maximum recursion depth exceeded
]
[autoreload of tensorflow_estimator._api.v1.estimator failed: Traceback (most recent call last):
  File "/usr/loca

## Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/drive


## Helper Classes: BERT Model


In [0]:
from BertModel import BertModel

## Helper Classes: Albert Model

In [0]:
from AlbertModel import AlbertModel

## Helper Classes: Embeddings Generator

In [0]:
from EmbeddingsGenerator import EmbeddingsGenerator

## Similarity Calculator

In [0]:
from SimilarityEstimator import SimilarityEstimator

## Similarity Server

In [0]:
from SimilarityServer import SimilarityServer

## PlayGround

The SimilarityServer class is a simple server that demonstrates the whole process from loading the files, generating the emebeddings and querying.

To instantiate the class, it requires an instance of embeddings generator and similarity estimator

In [0]:
embeddingsGenerator = EmbeddingsGenerator(delete_models=False)

In [0]:
simcal = SimilarityEstimator()

In [0]:
similarity_server = SimilarityServer(embeddingsGenerator, simcal)

The following csv file constains a few samples for demonstrations

In [0]:
input_csv = 'minyset.csv'

The method below takes the file, generates the embeddings, and indexes the similarities.

In [0]:
similarity_server.create_similarity_index(input_csv, model_names=['elmo', 'use'])

Embeddings Generator: Creating elmo embeddings..
Embeddings Generator: Reading chunk_0 sentences from minyset.csv
Embeddings Generator: Loading elmo pretrained model
Embeddings Generator: Reading chunk_1 sentences from minyset.csv
Embeddings Generator: elmo already loaded
Embeddings Generator: Reading chunk_2 sentences from minyset.csv
Embeddings Generator: elmo already loaded
Embeddings Generator: Done Writing elmo embeddings to /content/drive/My Drive/LUDHH_elmo_embeddings.h5

Embeddings Generator: Creating use embeddings..
Embeddings Generator: Reading chunk_0 sentences from minyset.csv
Embeddings Generator: Loading use pretrained model
Embeddings Generator: Reading chunk_1 sentences from minyset.csv
Embeddings Generator: use already loaded
Embeddings Generator: Reading chunk_2 sentences from minyset.csv
Embeddings Generator: use already loaded
Embeddings Generator: Done Writing use embeddings to /content/drive/My Drive/LUDHH_use_embeddings.h5


Similarity Estimator: Opening /conten

Now we query the server to return the nearest 5 users to user with key 4122, using the universal sentence encoder algorithm, and the cosine similarity measure

The second line contains the sentence of the queries user. And the table below shows the neares 5 with their ids and similarity scores.

In [0]:
similarity_server.get_most_similar_K_to(4122, 5, 'use', 'cosine');

two young women with folded arms are looking at the screen
----  --------  ----------------------------------------------------------------------
   1  0.126156  two women with folded arms , who are young , are looking at the screen
2925  0.620487  two children are crouching under some metal bars
  57  0.635718  a woman in a purple top is walking a bicycle between two stone columns
   3  0.641305  two children are crouching under some bars made of metal
  56  0.668657  a woman is putting cosmetics on her eyelid
----  --------  ----------------------------------------------------------------------


# Server

In [1]:
import json
import os
import random
import string
from flask import Flask, render_template, request, Response
from werkzeug.utils import secure_filename
from EmbeddingsGenerator import EmbeddingsGenerator
from SimilarityEstimator import SimilarityEstimator
from SimilarityWrapper import SimilarityWrapper

app = Flask(__name__)

## Constants
UPLOAD_FOLDER = 'static/uploads'
OUTPUT_DIR = os.path.abspath('embeddings')

## Setup
embeddingsGenerator = EmbeddingsGenerator(delete_models=False)
similarityEstimator = SimilarityEstimator(output_dir=OUTPUT_DIR)
similarity_server = SimilarityWrapper(embeddingsGenerator, similarityEstimator, output_dir=OUTPUT_DIR)
input_csv = os.path.abspath('static/uploads/minyset.csv')

## Renders the web page to upload the csv file
@app.route('/upload')
def upload():
   return render_template('upload.html')


## Uploads the csv file which will be used to generate the embeddings
@app.route('/uploader', methods = ['GET', 'POST'])
def upload_file():
   if request.method == 'POST':
      f = request.files['file']
      file_path = '{}/{}'.format(UPLOAD_FOLDER, secure_filename(f.filename))
      f.save(file_path)
      input_csv = file_path
      return 'file uploaded successfully'


## GET Request: Returns the list of supported models
@app.route('/models', methods = ['GET'])
def get_models():
    return json.dumps(similarity_server.get_available_embedding_algorithms())


## GET Request: Returns the list of supported distance metrics
@app.route('/metrics', methods = ['GET'])
def get_metrics():
    return json.dumps(similarity_server.get_available_distance_metrics())

## Post Request: Generates Embeddings and Similarity Indices
## Params:
##      models: list of strings, the models to generate the embeddings for.
##      metrics: list of strings, the metrics to use in calculating the distance.
##      max_k: integer, the number of maximum similar users to index.
##      chunk_size: integer, chunk size to use in reading data from csv file
##      id_key: str, name of the column with the user ids
##      sentence_key: str, name of the column with user objectives
@app.route('/generate_models', methods = ['POST'])
def generate_models():
    models = request.json['models']
    metrics = request.json['metrics']
    max_k = request.json['max_k']
    chunk_size = request.json['chunk_size']
    id_key = request.json['id_key']
    sentence_key = request.json['sentence_key']

    print('Generating {} for {}'.format(models, metrics))
    
    output_prefix = ''.join(random.choices(string.ascii_uppercase, k=5))

    embedding_files = similarity_server.generate_emeddings_from_csv(input_csv, OUTPUT_DIR, output_prefix, chunk_size, models, id_key, sentence_key)
    for embedding_algorithm, embedding_file in embedding_files.items():
        similarity_server.create_embeddings_index(embedding_file, embedding_algorithm, max_k, metrics)

    return Response(status=201)

## Post Request: Returns the objective of the given user is
## Params:
##      id: integer, the id of the user to return their objective
@app.route('/get_sentence', methods = ['POST'])
def get_sentence():
    return similarity_server.get_sentence_at_id(request.json['id'], input_csv)

## Post Request: Returns the most similar users to the given id
## Params:
##      models: string, the embedding technique to use.
##      metric: string, the metric to use in calculating the distance.
##      K: integer, the number of similar users to return.
##      id: integer, the id of the user in query
## Returns:
##      list of user ids, objectives, and scores
@app.route('/get_nearest', methods = ['POST'])
def get_nearest():
    method = request.json['method']
    metric = request.json['metric']
    id = request.json['id']
    K = request.json['K']
    __, nearest = similarity_server.get_most_similar_K_to(id, K, method, metric, input_csv)
    return nearest.to_json()


if __name__ == '__main__':
   app.run(debug = True)

ModuleNotFoundError: ignored